# Turtle Breakout 11 — one account across 30 coins: risk-allocation rules R0–R4, long only and long + reverse

Pre-registration: `discussions/Turtle_Breakout/013_claude_tb11_preregistration.md`. Code: `src/data_engineering/modeling/turtle_portfolio.py`.

All 30 coins share **one** account (equity marked daily at 17:00 New York). Base risk per trade $r = 2\%\cdot\mathrm{HWM}$.
R0 no rule · R1 heat cap $H$ · R2 correlation scaling · R3 cluster caps $C$ (K = 4) · R4 walk-forward coin weights.
**Discovery** 2020–2023 chooses $H$ and $C$ (max Calmar). **Evaluation** 2024-01-01 → end decides: Calmar(rule) − Calmar(R0m), R0m = R0 at the same mean open risk; block bootstrap, Holm over 6 (R1–R3 × 2 variants).

Environment: `TB01_DATA_DIR` (default `BINANCE_UM_DIR\long`), `TB11_SYNTHETIC=1`.

In [ ]:
import os, sys, json, gc, time, subprocess, tempfile, warnings, platform
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 150)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'data_engineering').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); load_dotenv(ROOT / '.env')
from data_engineering.modeling import turtle_breakout as tb, turtle_trade_features as tf, turtle_sizing as ts, turtle_reverse as tr, turtle_portfolio as tp

SYNTHETIC = os.getenv('TB11_SYNTHETIC', '0') == '1'
P = tb.Params(); PP = tp.PParams(); RP = tr.RParams()
DISC0, DISC1 = tp.DISC; EVAL0 = tp.EVAL_START
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='tb11-')) / '11'; PP = tp.PParams(n_boot=300, H_grid=(0.04, 0.08, 0.15), C_grid=(0.02, 0.06))
    DISC0, DISC1, EVAL0 = pd.Timestamp('2021-07-01', tz='UTC'), pd.Timestamp('2023-07-01', tz='UTC'), pd.Timestamp('2023-07-01', tz='UTC')
else:
    OUT = ROOT / 'notebooks' / 'Turtle_Breakout' / '11'
    base = Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent / 'Binance'
    DATA = Path(os.environ.get('TB01_DATA_DIR', str(base / 'long')))
OUT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy(); env['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + env.get('PYTHONPATH', '')
chk = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_turtle_portfolio.py'], cwd=str(ROOT), env=env, capture_output=True, text=True)
print(chk.stderr.strip().splitlines()[-1]); assert chk.returncode == 0
print(json.dumps(tp.registration(PP), indent=1, default=str))

## 1. Books: trades of every coin, long only and long + reverse

In [ ]:
BOOKS = {'long': {}, 'long_rev': {}}; COV = []
if SYNTHETIC:
    BOOKS['long'] = tp.synthetic_books(10, 1100, seed=3, with_reverse=False)
    BOOKS['long_rev'] = tp.synthetic_books(10, 1100, seed=3, with_reverse=True)
    COV = pd.DataFrame([{'asset': a, 'longs': len(b.trades), 'reverses': int((BOOKS['long_rev'][a].trades.kind == 'reverse').sum())} for a, b in BOOKS['long'].items()])
else:
    for a in tf.UNIVERSE:
        t1 = time.time()
        pu = DATA / f'{a}USDT_um_1m.parquet'
        if not pu.exists():
            print(a, 'missing'); continue
        b, _ = tf.load_um(pu)
        fpth = DATA / 'basis' / 'funding' / f'{a}USDT_funding.parquet'
        f = tb.funding_events(pd.read_parquet(fpth)) if fpth.exists() else None
        if a == 'TON':
            b = b[b.index < tr.TON_CUTOFF]
        D = tb.prepare_asset(a, b, f, P)
        start = b.index[0] + pd.Timedelta(days=30)
        L0 = ts.trade_list(D, P, start, RP, with_reverse=False); L1 = ts.trade_list(D, P, start, RP, with_reverse=True)
        BOOKS['long'][a] = tp.make_book(D, L0, P); BOOKS['long_rev'][a] = tp.make_book(D, L1, P)
        COV.append({'asset': a, 'start': start, 'end': b.index[-1], 'longs': len(L0), 'reverses': int((L1.kind == 'reverse').sum()), 'funding': f is not None})
        print(f'{a}: {len(L0)} longs, {COV[-1]["reverses"]} reverses, {time.time() - t1:.0f}s')
        del b, D; gc.collect()
    COV = pd.DataFrame(COV)
display(COV)
RET = tp.returns_panel(BOOKS['long'])
LAST = max(b.trades.t_out.max() for b in BOOKS['long'].values())
print('returns panel', RET.shape, '| last trade exit', LAST)

## 2. Discovery (2020–2023): grids for H (R1) and C (R3); every rule

In [ ]:
def run(variant, rule, t0, t1, pp=PP, books=None, **kw):
    o = tp.simulate(books or BOOKS[variant], rule, pp, t0, t1, R=RET, **kw)
    Dd = tp.daily_curve(o['equity'], pp.start_equity)
    return {**tp.metrics(Dd, pp.start_equity), 'max_open_risk': o['max_open_risk'], 'mean_open_risk': o['mean_open_risk'],
            'max_gross_lev': o['max_gross_lev'], **o['stats']}, Dd, o
T0 = time.time(); DISC_ROWS = []; SEL = {}
for v in ('long', 'long_rev'):
    for rule, kw in [('R0', {}), ('R2', {}), ('R4', {})] + [('R1', {'H': h}) for h in PP.H_grid] + [('R3', {'C': c}) for c in PP.C_grid]:
        m, _, _ = run(v, rule, DISC0, DISC1, **kw)
        DISC_ROWS.append({'variant': v, 'rule': rule, 'H': kw.get('H', np.nan), 'C': kw.get('C', np.nan), **m})
    print(v, f'{time.time() - T0:.0f}s')
DISC = pd.DataFrame(DISC_ROWS)
for v in ('long', 'long_rev'):
    g = DISC[DISC.variant == v]
    SEL[v] = {'H': tp.select(g[g.rule == 'R1'], 'H'), 'C': tp.select(g[g.rule == 'R3'], 'C')}
print('selected:', SEL)
display(DISC[['variant', 'rule', 'H', 'C', 'cagr', 'max_dd', 'calmar', 'sharpe', 'ruined', 'max_open_risk', 'mean_open_risk', 'max_gross_lev', 'entries', 'skipped', 'scaled']].round(3))

## 3. Evaluation (2024 → end): decision

In [ ]:
EV_ROWS, EV_CURVES, EV_OUT, TESTS = [], {}, {}, []
for v in ('long', 'long_rev'):
    for rule, kw in (('R0', {}), ('R1', {'H': SEL[v]['H']}), ('R2', {}), ('R3', {'C': SEL[v]['C']}), ('R4', {})):
        m, Dd, o = run(v, rule, EVAL0, None, **kw)
        EV_ROWS.append({'variant': v, 'rule': rule, 'param': kw.get('H', kw.get('C', np.nan)), **m}); EV_CURVES[(v, rule)] = Dd; EV_OUT[(v, rule)] = o
    r0 = EV_OUT[(v, 'R0')]['mean_open_risk']
    for rule in ('R1', 'R2', 'R3'):
        fm = PP.f * EV_OUT[(v, rule)]['mean_open_risk'] / r0
        m, Dd, o = run(v, 'R0', EVAL0, None, pp=replace(PP, f=fm))
        EV_ROWS.append({'variant': v, 'rule': f'R0m_for_{rule}', 'param': fm, **m}); EV_CURVES[(v, f'R0m_{rule}')] = Dd
        t = tp.calmar_diff(EV_CURVES[(v, rule)], Dd, PP, PP.seed + len(TESTS))
        t0 = tp.calmar_diff(EV_CURVES[(v, rule)], EV_CURVES[(v, 'R0')], PP, PP.seed + 100 + len(TESTS))
        TESTS.append({'variant': v, 'rule': rule, 'f_R0m': fm, **t, 'vs_R0_calmar_diff': t0['calmar_diff'], 'vs_R0_lo95': t0['lo95'], 'vs_R0_hi95': t0['hi95'], 'vs_R0_p': t0['p']})
EVAL = pd.DataFrame(EV_ROWS); TESTS = pd.DataFrame(TESTS)
TESTS['p_holm'] = tp.holm(TESTS.p.to_numpy())
TESTS['result'] = np.where(TESTS.p_holm < PP.alpha, 'IMPROVES', 'NO_DIFFERENCE')
display(EVAL[['variant', 'rule', 'param', 'cagr', 'max_dd', 'calmar', 'sharpe', 'ruined', 'max_open_risk', 'mean_open_risk', 'max_gross_lev', 'entries', 'skipped', 'scaled']].round(3))
display(TESTS.round(4))
DEC = {'selected': SEL, 'tests': TESTS[['variant', 'rule', 'calmar_diff', 'lo95', 'hi95', 'p', 'p_holm', 'result']].to_dict('records'),
       'any_improves': bool((TESTS.result == 'IMPROVES').any())}
DEC['verdict'] = 'RULE_IMPROVES: ' + ', '.join(f"{r.rule}/{r.variant}" for r in TESTS.itertuples() if r.result == 'IMPROVES') if DEC['any_improves'] else 'NO_RULE_BEATS_SAME_EXPOSURE'
print(json.dumps(DEC, indent=1, default=float))

## 4. Reported: long + reverse vs long under each rule, equal-weight sleeves, full period, 4-coin portfolio

In [ ]:
rows = []
for rule in ('R0', 'R1', 'R2', 'R3', 'R4'):
    t = tp.calmar_diff(EV_CURVES[('long_rev', rule)], EV_CURVES[('long', rule)], PP, PP.seed + 500)
    rows.append({'rule': rule, 'calmar_long': float(EVAL[(EVAL.variant == 'long') & (EVAL.rule == rule)].calmar.iloc[0]),
                 'calmar_long_rev': float(EVAL[(EVAL.variant == 'long_rev') & (EVAL.rule == rule)].calmar.iloc[0]), **t})
REV = pd.DataFrame(rows); print('Evaluation: long + reverse minus long (Calmar)'); display(REV.round(3))

# equal-weight sleeves: each coin its own account (TB 10 pool), same windows
SLV = []
for v in ('long', 'long_rev'):
    for name, (a0, a1) in (('discovery', (DISC0, DISC1)), ('evaluation', (EVAL0, None))):
        cur = {}
        for a, bk in BOOKS[v].items():
            o = tp.simulate({a: bk}, 'R0', PP, a0, a1)
            if len(o['equity']) > 30 and o['stats']['entries'] > 0:
                cur[a] = tp.daily_curve(o['equity'], PP.start_equity)
        lr = ts.pooled_log_returns(cur); Dd = PP.start_equity * np.exp(lr.cumsum())
        SLV.append({'variant': v, 'window': name, 'coins': len(cur), **tp.metrics(Dd, PP.start_equity)})
        if name == 'evaluation':
            EV_CURVES[(v, 'sleeves')] = Dd
SLV = pd.DataFrame(SLV); print('Equal-weight sleeves'); display(SLV.round(3))

FULL = []
for v in ('long', 'long_rev'):
    for rule, kw in (('R0', {}), ('R1', {'H': SEL[v]['H']}), ('R2', {}), ('R3', {'C': SEL[v]['C']}), ('R4', {})):
        m, _, _ = run(v, rule, DISC0, None, **kw); FULL.append({'variant': v, 'rule': rule, **m})
        if not SYNTHETIC:
            core = {a: BOOKS[v][a] for a in tf.CONFIRM_ASSETS if a in BOOKS[v]}
            m4, _, _ = run(v, rule, tb.COMMON_START, None, books=core, **kw); FULL.append({'variant': v, 'rule': rule + '_4coins', **m4})
FULL = pd.DataFrame(FULL); print('Full period (2020 -> end) and 4-coin portfolio (from 2020-10-15)')
display(FULL[['variant', 'rule', 'cagr', 'max_dd', 'calmar', 'sharpe', 'ruined', 'max_open_risk', 'mean_open_risk', 'max_gross_lev', 'entries', 'skipped']].round(3))

## 5. Figures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
colors = {'R0': 'k', 'R1': '#1f5aa6', 'R2': '#3a7d44', 'R3': '#c2571a', 'R4': '#8a4fa3', 'sleeves': '0.55'}
for j, v in enumerate(('long', 'long_rev')):
    for rule in ('R0', 'R1', 'R2', 'R3', 'R4', 'sleeves'):
        c = EV_CURVES[(v, rule)]
        axes[0, j].plot(c.index, c / PP.start_equity, color=colors[rule], lw=1.3 if rule != 'sleeves' else 1.0, ls='-' if rule != 'sleeves' else '--', label=rule)
        axes[1, j].plot(c.index, (c / c.cummax() - 1) * 100, color=colors[rule], lw=1.0, ls='-' if rule != 'sleeves' else '--', label=rule)
    axes[0, j].set_title(f'Evaluation equity, {v}', loc='left', fontsize=10); axes[1, j].set_title(f'Drawdown %, {v}', loc='left', fontsize=10)
    for a in axes[:, j]:
        a.grid(alpha=0.25); a.legend(frameon=False, fontsize=8, ncol=3)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(14, 3.5))
for rule in ('R0', 'R1', 'R2', 'R3'):
    s = EV_OUT[('long', rule)]['open_risk']; ax.plot(s.index, s * 100, color=colors[rule], lw=0.9, label=rule)
ax.set_ylabel('open risk, % of HWM'); ax.set_title('Evaluation, long only: open risk', loc='left', fontsize=10); ax.legend(frameon=False, fontsize=8); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 6. Save

In [ ]:
DISC.to_csv(OUT / '11_discovery.csv', index=False); EVAL.to_csv(OUT / '11_evaluation.csv', index=False); TESTS.to_csv(OUT / '11_tests.csv', index=False)
REV.to_csv(OUT / '11_reverse_vs_long.csv', index=False); SLV.to_csv(OUT / '11_sleeves.csv', index=False); FULL.to_csv(OUT / '11_full_and_4coins.csv', index=False)
COV.to_csv(OUT / '11_coverage.csv', index=False)
pd.DataFrame({f'{v}|{r}': c for (v, r), c in EV_CURVES.items()}).to_csv(OUT / '11_eval_curves.csv')
(OUT / '11_decision.json').write_text(json.dumps({'decision': DEC, 'registration': tp.registration(PP), 'synthetic': SYNTHETIC}, indent=1, default=str))
git = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=str(ROOT), capture_output=True, text=True).stdout.strip()
(OUT / '11_manifest.json').write_text(json.dumps({'version': tp.VERSION, 'git': git, 'python': platform.python_version(),
    'created_utc': pd.Timestamp.now('UTC').isoformat(), 'coins': list(BOOKS['long']), 'synthetic': SYNTHETIC}, indent=1))
print('saved to', OUT); print('VERDICT:', DEC['verdict'])